# 05 -- Inference

End-to-end offline prediction and the interactive predict+SHAP widget.

In [ ]:
import os
import sys
from pathlib import Path


def _find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "backend" / "app").is_dir() and (candidate / "data").is_dir():
            return candidate
        alt = candidate / "Olist_Marketplace_Platform"
        if (alt / "backend" / "app").is_dir() and (alt / "data").is_dir():
            return alt
    raise RuntimeError("Could not locate the project root above this notebook.")


PROJECT_ROOT = _find_project_root(Path.cwd())
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT / "backend"))
print("Project root:", PROJECT_ROOT)


In [ ]:
# Offline equivalent of a live health check + end-to-end prediction (no internet,
# no dependency on the hosted deployment being up -- see DEPLOYMENT.md for the
# live URLs if you do want to check the actual hosted system).
from app.services.model_registry import ModelRegistry
from app.services.sentiment_service import predict_sentiment

registry = ModelRegistry()
registry.load_all()
print("Loaded artifacts:", {k: v.status for k, v in registry.statuses.items()})

result = predict_sentiment(
    registry,
    "This item arrived broken and the seller never responded.",
    model_name="cnn2d",
)
print("\nEnd-to-end prediction (same code path the API's /sentiment/pipeline endpoint uses):")
print(result)


---
## 21. Interactive demo — try the model + SHAP explanation live

Type any review below, pick BERT or CNN2D, and click **Predict**. For BERT,
also shows a SHAP bar chart of which words pushed the prediction toward
Positive (green) or Negative (red) — the same `explain_single_review` used
by the live product's "explain" feature. Fully offline, in-process (no API
call, no internet) — reuses the models already validated throughout this
notebook.

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output
import matplotlib.pyplot as plt

from app.services.model_registry import ModelRegistry
from app.services.sentiment_service import predict_sentiment
from app.ml.explainability import is_shap_available, load_shap_explainer, explain_single_review
from app.ml.models import load_fine_tuned_bert
from app.ml.utils import get_device

print("Loading models for the interactive demo (BERT, CNN2D, and a SHAP explainer)...")
_demo_registry = ModelRegistry()
_demo_registry.load_all()

_device = get_device()
_bert_model, _bert_tokenizer = load_fine_tuned_bert("models/bert_review_sentiment", device=_device)
_shap_explainer = load_shap_explainer(_bert_model, _bert_tokenizer, device=-1) if is_shap_available() else None
print("Ready.")

text_box = widgets.Textarea(
    value="The product arrived broken and the seller never responded.",
    placeholder="Type or paste a review here...",
    layout=widgets.Layout(width="100%", height="80px"),
)
model_dropdown = widgets.Dropdown(options=["bert", "cnn2d"], value="bert", description="Model:")
explain_checkbox = widgets.Checkbox(value=True, description="Show SHAP explanation (BERT only)")
run_button = widgets.Button(description="Predict", button_style="primary")
output_area = widgets.Output()


def run_demo(text: str, model_name: str, show_explanation: bool):
    """Core logic behind the button click -- kept as a standalone function so it
    can also be called directly (e.g. for testing) without a live widget UI."""
    text = text.strip()
    if not text:
        print("Enter a review first.")
        return

    result = predict_sentiment(_demo_registry, text, model_name=model_name)
    print(f"Model: {model_name.upper()}")
    print(f"Prediction: {result['label']}  (confidence: {result['confidence']:.1%})")
    print(f"  P(Negative)={result['probability_negative']:.3f}  P(Positive)={result['probability_positive']:.3f}")

    if not show_explanation:
        return
    if model_name != "bert":
        print('\n(SHAP explanation only wired up for BERT here -- switch Model to "bert" to see it.)')
        return
    if _shap_explainer is None:
        print("\nSHAP not available (package not installed).")
        return

    explanation = explain_single_review(_shap_explainer, _bert_model, text, top_k=8)
    if not explanation["available"]:
        print("\nSHAP explanation unavailable:", explanation["reason"])
        return

    tokens = explanation["top_tokens_toward_positive"]
    labels = [t["token"] for t in tokens][::-1]
    values = [t["shap_value"] for t in tokens][::-1]
    colors = ["#2ca02c" if v >= 0 else "#c44e52" for v in values]

    fig, ax = plt.subplots(figsize=(6, 0.4 * len(labels) + 1))
    ax.barh(labels, values, color=colors)
    ax.axvline(0, color="black", linewidth=0.8)
    ax.set_xlabel("SHAP value (pushes toward Positive \u2192, \u2190 toward Negative)")
    ax.set_title("Top contributing tokens")
    plt.tight_layout()
    plt.show()


def _on_click(_):
    with output_area:
        clear_output()
        run_demo(text_box.value, model_dropdown.value, explain_checkbox.value)


run_button.on_click(_on_click)

display(widgets.VBox([
    widgets.HTML("<b>Try the model live</b>"),
    text_box,
    widgets.HBox([model_dropdown, explain_checkbox, run_button]),
    output_area,
]))
